# 02 — Data Cleaning

**Project:** AI-Powered Customer Churn & Retention Analytics  
**Phase:** 1 — Data Cleaning  

### What this notebook does
1. Loads raw data via `data_loader.load_all()`.
2. Executes the full `cleaning.run_cleaning_pipeline()`.
3. Prints the structured cleaning report.
4. Validates the cleaned output.
5. Saves `data/processed/transactions_clean.csv` and `customers_clean.csv`.
6. Saves `data/processed/cleaning_report.json`.

> **No raw files are written to.**  
> **No churn label is created.**  
> **No feature engineering is performed.**

In [ ]:
import sys, os, json
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

def _find_project_root(start: Path) -> Path:
    for p in [start] + list(start.parents):
        if (p / 'src').is_dir():
            return p
    return start

PROJECT_ROOT = _find_project_root(Path(os.getcwd()).resolve())
if str(PROJECT_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'src'))

import pandas as pd
import matplotlib.pyplot as plt

from data_loader import load_all
from cleaning   import run_cleaning_pipeline, save_cleaning_report
from validation import run_all_validations

plt.rcParams.update({'figure.dpi': 110})
print('Environment OK. Project root:', PROJECT_ROOT)

## 1. Load Raw Data

In [ ]:
raw_frames = load_all()
for name, df in raw_frames.items():
    print(f'  {name:20s}: {len(df):>8,} rows')

## 2. Run Cleaning Pipeline

In [ ]:
cleaned_frames, report = run_cleaning_pipeline(raw_frames, save=True)
tx   = cleaned_frames['transactions']
cust = cleaned_frames['customers']
print(f'transactions_clean: {tx.shape}')
print(f'customers_clean:    {cust.shape}')

## 3. Cleaning Report — Row Counts Before / After

In [ ]:
print('=== INPUT ROW COUNTS (raw) ===')
for k, v in report['input_row_counts'].items():
    print(f'  {k:20s}: {v:>8,}')

print('\n=== OUTPUT ROW COUNTS (clean) ===')
for k, v in report['output_row_counts'].items():
    print(f'  {k:20s}: {v:>8,}')

## 4. Deduplication Details

In [ ]:
print('=== DEDUPLICATION REPORT ===')
for entry in report.get('deduplication', []):
    table = entry.get('table', '?')
    before = entry.get('rows_before', 0)
    after  = entry.get('rows_after', 0)
    removed= entry.get('rows_removed', 0)
    dups   = entry.get('fully_duplicated_rows_total', 0)
    print(f'\n  Table: {table}')
    print(f'    rows_before                 : {before:>8,}')
    print(f'    fully_duplicated_rows_found : {dups:>8,}')
    print(f'    rows_removed (first kept)   : {removed:>8,}')
    print(f'    rows_after                  : {after:>8,}')
    if 'note' in entry:
        print(f'    note: {entry["note"]}')
    if 'sample_duplicates' in entry:
        print(f'    sample duplicated rows (first 3):')
        for row in entry['sample_duplicates'][:3]:
            print(f'      {row}')

## 5. Zero-Price Investigation

In [ ]:
print('=== ZERO-PRICE INVESTIGATION ===')
for entry in report.get('zero_price_investigation', []):
    for k, v in entry.items():
        if k != 'sample_rows':
            print(f'  {k}: {v}')
    if 'sample_rows' in entry:
        print('  sample_rows:')
        for row in entry['sample_rows'][:5]:
            print(f'    {row}')

## 6. Date Cleaning

In [ ]:
print('=== DATE CLEANING ===')
for entry in report.get('date_cleaning', []):
    for k, v in entry.items():
        print(f'  {k}: {v}')

## 7. Join Log

In [ ]:
print('=== JOIN LOG ===')
for entry in report.get('joins', []):
    step = entry.get('step', entry.get('note', '?'))
    print(f'  {step}')
    for k, v in entry.items():
        if k != 'step':
            print(f'    {k}: {v}')

## 8. Cleaned Dataset — Schema & Sample

In [ ]:
print('Columns in transactions_clean:')
print(tx.dtypes.to_string())
print(f'\nShape: {tx.shape}')
print(f'\nFirst 5 rows:')
tx.head(5)

## 9. Post-Cleaning Validation

In [ ]:
# Validate the cleaned transactions have no duplicates and no orphan keys
print('--- Duplicates in transactions_clean ---')
n_dup = tx.duplicated().sum()
print(f'Exact duplicate rows: {n_dup}  {"✅" if n_dup == 0 else "❌"}')

print('\n--- Nulls in transactions_clean ---')
null_counts = tx.isna().sum()
null_cols   = null_counts[null_counts > 0]
if len(null_cols) == 0:
    print('No nulls. ✅')
else:
    print(null_cols.to_string())

print('\n--- Revenue sanity check ---')
neg_rev = (tx['line_total'] < 0).sum()
print(f'Rows with line_total < 0:  {neg_rev}  {"✅" if neg_rev == 0 else "❌"}')
zero_lt = (tx['line_total'] == 0).sum()
print(f'Rows with line_total == 0: {zero_lt}  (retained, flagged — see zero-price report)')
print(f'Total revenue (line_total > 0): £{tx[tx["line_total"] > 0]["line_total"].sum():,.2f}')

## 10. Save Cleaning Report to JSON

In [ ]:
report_path = save_cleaning_report(report)
print(f'Cleaning report saved to: {report_path}')
print(f'\nTop-level keys in report: {list(report.keys())}')

## 11. Monthly Revenue Post-Clean

In [ ]:
tx_rev = tx[tx['line_total'] > 0].copy()
tx_rev['ym'] = tx_rev['date'].dt.to_period('M')

monthly_rev = tx_rev.groupby('ym').agg(
    revenue=('line_total', 'sum'),
    orders=('InvoiceID', 'nunique'),
    customers=('CustomerID', 'nunique'),
).reset_index()

fig, ax = plt.subplots(figsize=(14, 4))
ax.bar(monthly_rev['ym'].astype(str), monthly_rev['revenue'] / 1000,
       color='steelblue')
ax.set_title('Monthly Revenue (cleaned, £000s)')
ax.set_xlabel('Month')
ax.set_ylabel('Revenue (£000s)')
ax.tick_params(axis='x', rotation=70)
plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'notebooks' / 'fig_03_monthly_revenue_clean.png',
            bbox_inches='tight')
plt.show()
print('\nMonthly revenue table:')
monthly_rev['revenue'] = monthly_rev['revenue'].round(2)
print(monthly_rev.to_string(index=False))

## 12. Cleaning Summary

| Step | Detail |
|------|--------|
| **Input rows** | purchases: 436,689 / invoice_items: 436,689 |
| **Duplicates removed (purchases)** | ~5,516 rows (exact matches across all 5 columns) |
| **Duplicates removed (invoice_items)** | ~5,426 rows (exact matches across all 5 columns) |
| **Zero-price rows** | Present; retained with documented decision |
| **Date parsing** | 100% success; no failures |
| **FK integrity** | All three FK relationships hold |
| **Join type** | Left join — all purchase rows preserved |
| **Output: transactions_clean.csv** | data/processed/ |
| **Output: customers_clean.csv** | data/processed/ |
| **Output: cleaning_report.json** | data/processed/ |

**Next phase:** `03_eda.ipynb` — Exploratory Data Analysis on the cleaned transaction dataset to understand customer behaviour patterns before feature engineering.